# Recs 004: Offline eval — same-user held-out likes (proxy task)

## Key Goal

Measure offline ranking quality on a same-user proxy task to compare query construction methods.

## Decision It Supports

Which query method should be default under this proxy objective.

## Primary Metrics

Hit@K, Recall@K, MAP@K, NDCG@K, MRR (with notebook-defined aggregation semantics).

## Run feedback

On a representative **val** proxy run with this notebook’s defaults, the **§3** summary table often looks like:

- **`popularity_train`** is **strongest** on Hit / Recall / MAP / NDCG / MRR — global **train** thumbs-up counts are hard to beat when the indexed catalog is modest and this eval slice lines up with “what’s popular.” Beating this baseline is a **separate milestone** from beating random.
- **`random`** is **weakest** (sanity check).
- **Embedding rows** (`raw`, `structured`, `multi_*`, `tw_*`) usually sit **between** random and popularity; with rules-based `extract_preferences`, **`raw`** often edges **`structured`**.
- **`tw_train_mean_30d`** and **`tw_train_mean_365d`** are often **nearly identical** — many users have little qualifying train text in one window but not the other, or recency weighting washes out the gap.
    - **`tw` means "time-weighted"!!!

Re-run after changing **`MAX_USERS`**, eval split, or data; this is **one cohort** (multi-review val users), not “all Steam.”

## Metrics (§3 summary table)

Same definitions as the **§3** helpers: **binary** relevance; **positives** = other **eval-split** liked games for that user (excluding the query game). Values are **per user**, then aggregated in `summarize`: **Hit@K** uses **mean**; **Recall@K**, **MAP@K**, **NDCG@K**, and **MRR** use **nanmean** (so users with no positives do not pollute those averages).

| Metric | Definition |
|--------|------------|
| **Hit@K** | **1** if **at least one** positive appears in the top-**K** ranked games, else **0**. Mean over users → fraction of users with any hit in top-K (sometimes called success@K). |
| **Recall@K** | **(positives that appear in top‑K) ÷ (all positives for that user)** — fraction of their relevant games that land in the top‑K list. Users with many positives get a larger denominator (harder). |
| **MAP@K** | **Mean average precision** truncated at K: walk ranks **1…K**; each time you see a positive, add **(hits so far) / (current rank)**; divide that sum by the user’s **positive count** to get one AP per user; average across users. |
| **NDCG@K** | **Normalized DCG** at K: relevance **1** on positives, **0** otherwise; **DCG** = sum of **rel ÷ log₂(rank + 1)** (rank 1-based in code); **NDCG** = DCG ÷ **ideal DCG** for that user’s positive count; average across users. |
| **MRR** | **Mean reciprocal rank** of the **first** positive in the **full** ranked list (**1/rank**), or **0** if none; average across users. |

### Method glossary (§3 rows)

| Method row | What it means |
|---|---|
| **random** | Random score for each candidate game (query game masked). Sanity-floor baseline. |
| **popularity_train** | Rank by global train-split thumbs-up count per game (`recommended == 1`), masking query game. Popularity-only baseline. |
| **raw** | Embed the single query review text directly (raw user language) and rank by cosine similarity to game-profile vectors. |
| **structured** | Run `extract_preferences` first, then embed the structured rewrite of the same query text before ranking. |
| **multi_mean_train** | Build one query vector by averaging embeddings of: query text + up to `MULTI_MAX_REVIEWS-1` prior **train** support reviews for that user. |
| **multi_concat_train** | Concatenate query text with selected **train** support reviews (char-capped), embed once, then rank. |
| **tw_train_mean_30d** | Time-weighted blend of query text plus same-user **train** support reviews within last 30 days before query time (more recent gets higher weight). |
| **tw_train_mean_365d** | Same as above but with a 365-day lookback window. |

**Sections:** **§1** setup · **§2** sample (**val** queries/labels + **train** history) · **§3** **ablation** — baselines, raw/structured, **train-pool** multi, **time-weighted train** (30d / 365d).

**Goal:** Compare **raw** vs **structured** query embedding on a concrete relevance definition: other games the **same** user thumbs-up reviewed (`recommended == 1`), excluding the query game. See **`docs/recommender_transition_plan.md`** → *Offline proxy task: other games the same user liked*.

**Requires:** [`recs_002`](./recs_002_game_embeddings_raw.ipynb) artifacts (`game_profile_embeddings.npz`, index Parquet, `meta.json`). Uses the **same TF Hub** model as `recs_003`.

**Test holdout:** set env **`RECS004_EVAL_SPLIT=test`** (and ensure `*_test_norm.parquet` exists) for a **one-shot** run after you freeze the method; default is **val**.

**Split choice:** Query reviews come from the **validation** split (`*_val_norm.parquet`) so you do not burn the **test** holdout while iterating. Game vectors are still built from **train** (`recs_001` / `recs_002`). If val is missing, the notebook falls back to **train** and prints a warning (query text may overlap game-profile pools).

**Caveats:** Users with only one indexed thumbs-up review are skipped. Possible **franchise correlation** among positives. **Recall@K** is normalized by \|positives\| so users with many likes have harder scores.

**Population / selection:** Examples are **multi-game-like users** in val only; they can **differ systematically** from single-review or non-reviewing users (engagement, taste breadth, text style). Do **not** treat these metrics as a population average for “all Steam users.” See **`docs/recommender_transition_plan.md`** → *Selection bias: multi-review vs single-review users*.

**Empirical note (val proxy, rules-based `extract_preferences`):** **raw** embedding usually **beats** **structured** on Hit@K / Recall@K / MRR here. Treat **raw** as the **default** query for USE + this eval until structured improves; keep structured as an **ablation** (see `docs/recommender_transition_plan.md`).

**On a small catalog, our same-user proxy correlates with popularity; we report a popularity baseline and treat beating it as a separate milestone — raw text similarity is not yet personalized enough.**

**Train-pool rows:** extra text for **`multi_*_train`** / **`tw_train_mean_*`** comes from **train** only (excluding val **positives ∪ query**; train rows with `timestamp_created` **> query_ts** dropped).


## 1) Paths, game matrix, embedder


In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# Repo root (same pattern as recs_003)
def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")


REPO_ROOT = _repo_root()
PROCESSED = REPO_ROOT / "data" / "processed"
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
NPZ_PATH = ARTIFACT_DIR / "game_profile_embeddings.npz"
INDEX_PATH = ARTIFACT_DIR / "game_profile_embedding_index.parquet"
META_PATH = ARTIFACT_DIR / "game_profile_embedding_meta.json"

VAL_PARQUET = PROCESSED / "steam_reviews_cleaned_english_val_norm.parquet"
TRAIN_PARQUET = PROCESSED / "steam_reviews_cleaned_english_train_norm.parquet"
TEST_PARQUET = PROCESSED / "steam_reviews_cleaned_english_test_norm.parquet"

# Default: val for iteration. One-shot final eval: RECS004_EVAL_SPLIT=test
_split = os.environ.get("RECS004_EVAL_SPLIT", "val").strip().lower()
if _split == "test":
    if not TEST_PARQUET.is_file():
        raise FileNotFoundError(
            f"RECS004_EVAL_SPLIT=test but missing {TEST_PARQUET} — run normalization (usage_pipeline.md)"
        )
    EVAL_PARQUET = TEST_PARQUET
    EVAL_SPLIT_NAME = "test"
elif _split == "train":
    EVAL_PARQUET = TRAIN_PARQUET if TRAIN_PARQUET.is_file() else None
    EVAL_SPLIT_NAME = "train"
    if EVAL_PARQUET is None:
        raise FileNotFoundError("train_norm parquet missing")
else:
    EVAL_PARQUET = VAL_PARQUET if VAL_PARQUET.is_file() else TRAIN_PARQUET
    EVAL_SPLIT_NAME = "val" if EVAL_PARQUET == VAL_PARQUET else "train"

for pth in (NPZ_PATH, INDEX_PATH, META_PATH):
    if not pth.is_file():
        raise FileNotFoundError(f"Run recs_002 first. Missing {pth}")
if not EVAL_PARQUET.is_file():
    raise FileNotFoundError(f"Missing {EVAL_PARQUET} — run normalization pipeline (see docs/usage_pipeline.md)")
if not TRAIN_PARQUET.is_file():
    raise FileNotFoundError(f"Missing {TRAIN_PARQUET} — needed for popularity baseline")

meta = json.loads(META_PATH.read_text(encoding="utf-8"))
TFHUB_URL = meta["model_name"]
EMBED_DIM = int(meta["dim"])
MAX_CHARS = meta.get("max_chars_per_review")

print("Eval split:", EVAL_SPLIT_NAME, "→", EVAL_PARQUET.name, "| RECS004_EVAL_SPLIT=", repr(_split))
print("TF Hub:", TFHUB_URL, "| dim:", EMBED_DIM)
if EVAL_SPLIT_NAME == "test":
    print("TEST holdout run — use only after freezing method; val is default when env unset.")


Eval split: val → steam_reviews_cleaned_english_val_norm.parquet | RECS004_EVAL_SPLIT= 'val'
TF Hub: https://tfhub.dev/google/universal-sentence-encoder/4 | dim: 512


In [2]:
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
import tensorflow_hub as hub

for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

embed_fn = hub.load(TFHUB_URL)


2026-04-11 13:54:26.772769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775930066.792540   51203 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775930066.798793   51203 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775930066.812588   51203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775930066.812610   51203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775930066.812612   51203 computation_placer.cc:177] computation placer alr

In [3]:
z = np.load(NPZ_PATH)
X = np.asarray(z["embeddings"], dtype=np.float32)
app_ids_X = np.asarray(z["app_id"], dtype=np.int64)
z.close()

idx_df = pd.read_parquet(INDEX_PATH)
if len(idx_df) != X.shape[0] or not np.array_equal(idx_df["app_id"].to_numpy(), app_ids_X):
    raise ValueError("Index / npz app_id alignment")

# row i <-> app_ids_X[i]
app_to_row = {int(a): i for i, a in enumerate(app_ids_X)}
indexed_apps = set(app_to_row.keys())
n_games = X.shape[0]
print("X:", X.shape, "unique games in index:", n_games)


X: (315, 512) unique games in index: 315


In [4]:
import math

def l2_normalize(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).ravel()
    nrm = np.linalg.norm(v)
    if nrm <= 1e-12:
        return v
    return (v / nrm).astype(np.float32)


def embed_text(text: str) -> np.ndarray:
    t = (text or "").strip()
    if MAX_CHARS is not None:
        t = t[: int(MAX_CHARS)]
    out = embed_fn([t])
    return l2_normalize(out)


def scores_excluding_query(q: np.ndarray, query_app_id: int) -> np.ndarray:
    s = (X @ q).astype(np.float32)
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def rank_app_ids(s: np.ndarray) -> np.ndarray:
    """Indices into X rows, highest score first."""
    return np.argsort(-s)


def recall_at_k(ranked_rows: np.ndarray, positives: set[int], k: int) -> float:
    top = set(int(app_ids_X[i]) for i in ranked_rows[:k])
    if not positives:
        return float("nan")
    return len(top & positives) / len(positives)


def hit_rate_at_k(ranked_rows: np.ndarray, positives: set[int], k: int) -> float:
    top = set(int(app_ids_X[i]) for i in ranked_rows[:k])
    return 1.0 if (top & positives) else 0.0


def mrr(ranked_rows: np.ndarray, positives: set[int]) -> float:
    for rank, i in enumerate(ranked_rows.tolist(), start=1):
        if int(app_ids_X[i]) in positives:
            return 1.0 / rank
    return 0.0


def average_precision_at_k(ranked_rows: np.ndarray, positives: set[int], k: int) -> float:
    """Binary relevance: average precision truncated at *k*, normalized by |positives|."""
    if not positives:
        return float("nan")
    hits = 0
    prec_sum = 0.0
    for rank, i in enumerate(ranked_rows[:k].tolist(), start=1):
        if int(app_ids_X[i]) in positives:
            hits += 1
            prec_sum += hits / rank
    return prec_sum / len(positives)


def ndcg_at_k(ranked_rows: np.ndarray, positives: set[int], k: int) -> float:
    """Binary NDCG@k: relevance 1 for positives in the top-*k* list."""
    if not positives:
        return float("nan")
    gains = [1.0 if int(app_ids_X[i]) in positives else 0.0 for i in ranked_rows[:k]]

    def dcg(g: list[float]) -> float:
        return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(g))

    ideal_len = min(len(positives), k)
    ideal_gains = [1.0] * ideal_len + [0.0] * max(0, k - ideal_len)
    idcg = dcg(ideal_gains)
    if idcg <= 1e-12:
        return 0.0
    return dcg(gains) / idcg


## 2) Sample eval rows (multi-review users)

**Split hygiene:** **Queries and labels** come from the **eval Parquet** chosen in §1 (default **`val_norm`**; **`test_norm`** if `RECS004_EVAL_SPLIT=test`). **Train-history text** for multi/time ablations comes from **`steam_reviews_cleaned_english_train_norm.parquet`** only. For each user we **block** train reviews whose `app_id` is the **val query game** or any **val positive** (so train text cannot literally describe a game you are trying to hit in val). Train rows with `timestamp_created` **after** the chosen val query time are dropped (causal filter w.r.t. that query).

Tune **`RNG_SEED`**, **`MAX_USERS`**, **`MIN_REVIEW_CHARS`**, **`MAX_TRAIN_ROWS_PER_USER`**.


In [5]:
from IPython.display import display

from steam_review_ml.recommender import build_embedding_input, extract_preferences

USER_COL = "author.steamid"
MIN_REVIEW_CHARS = 30
RNG_SEED = 42
MAX_USERS = 5000  # cap for notebook runtime; raise for stabler metrics
TIME_COL = "timestamp_created"
MAX_TRAIN_ROWS_PER_USER = 200  # cap stored train rows per user (after filtering)

rng = np.random.default_rng(RNG_SEED)

usecols = [USER_COL, "app_id", "review", "recommended", "review_id", TIME_COL]


def _load_split_df(path: Path) -> pd.DataFrame:
    d = pd.read_parquet(path, columns=usecols)
    if TIME_COL not in d.columns:
        raise KeyError(f"{path} missing {TIME_COL!r} — needed for time-weighted ablations")
    d = d.loc[d["recommended"] == 1].copy()
    d["review"] = d["review"].fillna("").astype(str)
    d = d[d["review"].str.len() >= MIN_REVIEW_CHARS]
    d = d[d["app_id"].isin(indexed_apps)]
    d["ts"] = pd.to_numeric(d[TIME_COL], errors="coerce")
    d = d.dropna(subset=["ts"])
    d["ts"] = d["ts"].astype(np.float64)
    return d


df_val = _load_split_df(EVAL_PARQUET)
df_train = _load_split_df(TRAIN_PARQUET)

if EVAL_SPLIT_NAME != "val":
    print(
        "WARNING: eval split is not val — train-vs-val profile hygiene is best defined when queries/labels are val. Prefer steam_reviews_cleaned_english_val_norm.parquet."
    )

uc = df_val.groupby(USER_COL)["app_id"].nunique()
multi = uc[uc >= 2].index
multi = pd.Index(rng.permutation(multi.values)[: min(len(multi), MAX_USERS)])

examples: list[dict] = []
for uid in multi:
    sub_v = df_val[df_val[USER_COL] == uid]
    sub_t = df_train[df_train[USER_COL] == uid]
    apps = sub_v["app_id"].unique().tolist()
    q_app = int(rng.choice(apps))
    rows_q = sub_v[sub_v["app_id"] == q_app]
    qi = rng.integers(0, len(rows_q))
    row_q = rows_q.iloc[qi]
    query_text = str(row_q["review"])
    query_ts = float(row_q["ts"])
    positives = {int(a) for a in apps if int(a) != q_app}
    blocklist = positives | {q_app}

    train_review_rows: list[dict] = []
    for _, r in sub_t.iterrows():
        aid = int(r["app_id"])
        ts = float(r["ts"])
        if aid in blocklist:
            continue
        if ts > query_ts:
            continue
        train_review_rows.append({"app_id": aid, "text": str(r["review"]), "ts": ts})
    rng.shuffle(train_review_rows)
    if len(train_review_rows) > MAX_TRAIN_ROWS_PER_USER:
        train_review_rows = train_review_rows[:MAX_TRAIN_ROWS_PER_USER]

    support_texts_train = [x["text"] for x in train_review_rows]

    examples.append(
        {
            "steamid": uid,
            "query_app_id": q_app,
            "query_text": query_text,
            "query_ts": query_ts,
            "positives": positives,
            "n_pos": len(positives),
            "support_texts_train": support_texts_train,
            "train_review_rows": train_review_rows,
        }
    )

print(f"Examples: {len(examples)} users (multi-review val, thumbs-up, in index)")
print("Positives per example: min", min(e["n_pos"] for e in examples), "max", max(e["n_pos"] for e in examples))


Examples: 5000 users (multi-review val, thumbs-up, in index)
Positives per example: min 1 max 8


## 3) Ablation: query variants vs baselines

One table; **all** retrieval rows use the **same val positives** (`positives` — other val liked games except the masked query game).

**Metrics:** **HitRate@K**, **Recall@K**, **MRR**, **MAP@K**, **NDCG@K** (binary positives).

| Method | Query vector | Notes |
|--------|--------------|-------|
| **random** | — | |
| **popularity_train** | — | |
| **raw** | embed val **query** review | |
| **structured** | prefs from val query | |
| **multi_mean_train** | mean: val query + **`support_texts_train`** from **train** only (apps **not** in val positives ∪ query); train rows **≤ query_ts** | |
| **multi_concat_train** | concat val query + train supports (capped) | |
| **tw_train_mean_30d** | exp-weighted mean: val query + **train** rows in **(query_ts−30d, query_ts]** (blocked apps) | |
| **tw_train_mean_365d** | same, **365d** window | |

Tune **`MULTI_MAX_REVIEWS`**, **`MULTI_CONCAT_CHARS`**, **`TAU_RECENCY_SEC`**, **`MAX_TRAIN_ROWS_PER_USER`** (§2).


In [6]:
# §3 — For each ablation method, aggregate Hit/Recall/MRR/MAP/NDCG over ``examples``.
KS = (5, 10, 20)
MULTI_MAX_REVIEWS = 5
MULTI_CONCAT_CHARS = 2000
TAU_RECENCY_SEC = 7 * 24 * 3600.0
SEC_PER_DAY = 86400.0
WINDOW_30D = 30 * SEC_PER_DAY
WINDOW_365D = 365 * SEC_PER_DAY

_train_use = ["app_id", "recommended"]
_df_tr = pd.read_parquet(TRAIN_PARQUET, columns=_train_use)
_df_tr = _df_tr.loc[_df_tr["recommended"] == 1]
_vc = _df_tr.groupby("app_id").size()
pop_row = np.asarray([float(_vc.get(int(a), 0)) for a in app_ids_X], dtype=np.float32)
pop_row = np.maximum(pop_row, 1e-6)


def summarize(name: str, agg: dict) -> pd.Series:
    """Collapse per-example lists in *agg* to one row for the summary table."""
    out = {}
    for k, v in agg.items():
        a = np.asarray(v, dtype=np.float64)
        if k.startswith("recall") or k.startswith("map") or k == "mrr" or k.startswith("ndcg"):
            out[k] = float(np.nanmean(a))
        else:
            out[k] = float(a.mean())
    return pd.Series(out, name=name)


def eval_loop(q_for_ex) -> dict:
    """Evaluate one query builder: embed, rank, append list metrics per example."""
    agg = (
        {f"hit@{k}": [] for k in KS}
        | {f"recall@{k}": [] for k in KS}
        | {f"map@{k}": [] for k in KS}
        | {f"ndcg@{k}": [] for k in KS}
        | {"mrr": []}
    )
    for ex in examples:
        pos = ex["positives"]
        if not pos:
            continue
        q = q_for_ex(ex)
        s = scores_excluding_query(q, ex["query_app_id"])
        order = rank_app_ids(s)
        for kk in KS:
            agg[f"hit@{kk}"].append(hit_rate_at_k(order, pos, kk))
            agg[f"recall@{kk}"].append(recall_at_k(order, pos, kk))
            agg[f"map@{kk}"].append(average_precision_at_k(order, pos, kk))
            agg[f"ndcg@{kk}"].append(ndcg_at_k(order, pos, kk))
        agg["mrr"].append(mrr(order, pos))
    return agg


def eval_random_baseline() -> dict:
    """Random scores; query ``app_id`` masked."""
    agg = (
        {f"hit@{k}": [] for k in KS}
        | {f"recall@{k}": [] for k in KS}
        | {f"map@{k}": [] for k in KS}
        | {f"ndcg@{k}": [] for k in KS}
        | {"mrr": []}
    )
    for ex in examples:
        pos = ex["positives"]
        if not pos:
            continue
        s = rng.random(n_games).astype(np.float32)
        row = app_to_row.get(int(ex["query_app_id"]))
        if row is not None:
            s[row] = -np.inf
        order = np.argsort(-s)
        for kk in KS:
            agg[f"hit@{kk}"].append(hit_rate_at_k(order, pos, kk))
            agg[f"recall@{kk}"].append(recall_at_k(order, pos, kk))
            agg[f"map@{kk}"].append(average_precision_at_k(order, pos, kk))
            agg[f"ndcg@{kk}"].append(ndcg_at_k(order, pos, kk))
        agg["mrr"].append(mrr(order, pos))
    return agg


def eval_popularity_baseline() -> dict:
    """Train popularity counts; query game masked."""
    agg = (
        {f"hit@{k}": [] for k in KS}
        | {f"recall@{k}": [] for k in KS}
        | {f"map@{k}": [] for k in KS}
        | {f"ndcg@{k}": [] for k in KS}
        | {"mrr": []}
    )
    for ex in examples:
        pos = ex["positives"]
        if not pos:
            continue
        s = pop_row.copy()
        row = app_to_row.get(int(ex["query_app_id"]))
        if row is not None:
            s[row] = -np.inf
        order = np.argsort(-s)
        for kk in KS:
            agg[f"hit@{kk}"].append(hit_rate_at_k(order, pos, kk))
            agg[f"recall@{kk}"].append(recall_at_k(order, pos, kk))
            agg[f"map@{kk}"].append(average_precision_at_k(order, pos, kk))
            agg[f"ndcg@{kk}"].append(ndcg_at_k(order, pos, kk))
        agg["mrr"].append(mrr(order, pos))
    return agg


def q_multi_mean_train(ex: dict) -> np.ndarray:
    """Query vector: L2-normalized mean of USE embeddings (val ``query_text`` + train supports from §2)."""
    texts = [ex["query_text"].strip()]
    for t in ex.get("support_texts_train", [])[: max(0, MULTI_MAX_REVIEWS - 1)]:
        if t and t.strip():
            texts.append(t.strip())
    if len(texts) == 1:
        return embed_text(texts[0])
    vecs = np.stack([embed_text(t) for t in texts], axis=0).astype(np.float32)
    return l2_normalize(vecs.mean(axis=0))


def q_multi_concat_train(ex: dict) -> np.ndarray:
    """Query vector: single USE embed on val query concatenated with train supports (capped)."""
    parts = [ex["query_text"].strip()]
    for t in ex.get("support_texts_train", []):
        if len(parts) >= MULTI_MAX_REVIEWS:
            break
        if t and t.strip():
            parts.append(t.strip())
        if sum(len(p) + 2 for p in parts) >= MULTI_CONCAT_CHARS:
            break
    blob = " \n\n ".join(parts)[:MULTI_CONCAT_CHARS]
    return embed_text(blob)


def q_tw_train_mean(ex: dict, window_sec: float) -> np.ndarray:
    """Weighted USE blend: val query plus train rows in the time window (§2 blocking)."""
    qt = float(ex["query_ts"])
    t_min = qt - window_sec
    rows = [r for r in ex["train_review_rows"] if t_min < r["ts"] <= qt]
    if not rows:
        return embed_text(ex["query_text"])
    wts = [1.0]
    vecs = [embed_text(ex["query_text"])]
    for r in rows:
        age = max(0.0, qt - r["ts"])
        w = float(np.exp(-age / TAU_RECENCY_SEC))
        wts.append(w)
        vecs.append(embed_text(r["text"]))
    wts = np.asarray(wts, dtype=np.float64)
    wts /= wts.sum() + 1e-12
    stacked = np.stack(vecs, axis=0).astype(np.float32)
    blended = (stacked * wts[:, None]).sum(axis=0)
    return l2_normalize(blended)


def q_raw(ex: dict) -> np.ndarray:
    """USE embedding of the val split ``query_text`` only (default v1 query)."""
    return embed_text(ex["query_text"])


def q_structured(ex: dict) -> np.ndarray:
    """USE embedding of ``build_embedding_input(extract_preferences(query_text), query_text)``."""
    return embed_text(build_embedding_input(extract_preferences(ex["query_text"]), ex["query_text"]))


agg_rand = eval_random_baseline()
agg_pop = eval_popularity_baseline()
agg_raw = eval_loop(q_raw)
agg_struct = eval_loop(q_structured)
agg_m_tr = eval_loop(q_multi_mean_train)
agg_m_cat_tr = eval_loop(q_multi_concat_train)
agg_tw30 = eval_loop(lambda ex: q_tw_train_mean(ex, WINDOW_30D))
agg_tw365 = eval_loop(lambda ex: q_tw_train_mean(ex, WINDOW_365D))

summary_ablation = pd.DataFrame(
    [
        summarize("random", agg_rand),
        summarize("popularity_train", agg_pop),
        summarize("raw", agg_raw),
        summarize("structured", agg_struct),
        summarize("multi_mean_train", agg_m_tr),
        summarize("multi_concat_train", agg_m_cat_tr),
        summarize("tw_train_mean_30d", agg_tw30),
        summarize("tw_train_mean_365d", agg_tw365),
    ]
)
display(summary_ablation.T)

print("Split:", EVAL_SPLIT_NAME, "| n_examples:", len(examples), "| n_games:", n_games)
if EVAL_SPLIT_NAME == "train":
    print("Note: queries are from TRAIN — train-profile vs val-label logic is weak. Prefer val or test.")
elif EVAL_SPLIT_NAME == "val":
    print("Validation queries — iterate here; set RECS004_EVAL_SPLIT=test for frozen one-shot.")
elif EVAL_SPLIT_NAME == "test":
    print("TEST holdout — document these numbers as final offline report for this method version.")


,random,popularity_train,raw,structured,multi_mean_train,multi_concat_train,tw_train_mean_30d,tw_train_mean_365d
hit@5,0.019800,0.126400,0.049800,0.029200,0.046200,0.045800,0.050200,0.050200
hit@10,0.042200,0.227600,0.089000,0.051400,0.089600,0.083200,0.089200,0.089400
hit@20,0.078200,0.371800,0.142600,0.089000,0.154000,0.139200,0.144000,0.144000
recall@5,0.015120,0.108053,0.040240,0.022350,0.036275,0.037373,0.040585,0.040585
recall@10,0.033475,0.196877,0.072082,0.040323,0.072385,0.067632,0.071539,0.071739
recall@20,0.063518,0.323938,0.116875,0.070928,0.125494,0.114734,0.118199,0.118199
map@5,0.006485,0.057204,0.020042,0.011403,0.017771,0.018136,0.020442,0.020442
map@10,0.008931,0.069002,0.024273,0.013790,0.022656,0.022246,0.024524,0.024543
map@20,0.011012,0.078253,0.027428,0.015945,0.026402,0.025531,0.027847,0.027847
ndcg@5,0.009207,0.072384,0.026362,0.015020,0.023735,0.024020,0.026808,0.026808


Split: val | n_examples: 5000 | n_games: 315
Validation queries — iterate here; set RECS004_EVAL_SPLIT=test for frozen one-shot.
